In [1]:
# import 
import pandas as pd
import numpy as np
import sqlite3


In [2]:
warehouses = pd.read_csv('C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/warehouses.csv' )
warehouses.head()

,warehouse_id,warehouse_name,city,state,region,capacity
0,WH032,Faridabad Distribution Center,Faridabad,Haryana,North,18934.0
1,WH006,Chennai Distribution Center,Chennai,Tamil Nadu,South,38118.0
2,WH033,Jaipur Distribution Center,Jaipur,Rajasthan,West,47129.0
3,WH014,Kanpur Distribution Center,Kanpur,Uttar Pradesh,North,22335.0
4,WH020,Coimbatore Distribution Center,Coimbatore,Tamil Nadu,South,10229.0


In [5]:
# checking the database
warehouses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   warehouse_id    55 non-null     object 
 1   warehouse_name  55 non-null     object 
 2   city            52 non-null     object 
 3   state           55 non-null     object 
 4   region          55 non-null     object 
 5   capacity        52 non-null     float64
dtypes: float64(1), object(5)
memory usage: 2.7+ KB


In [8]:
warehouses[warehouses['city'].isna()== True]

,warehouse_id,warehouse_name,city,state,region,capacity
19,WH028,Ahmedabad Distribution Center,NaN,Gujarat,West,49866.0
35,WH041,mysore distribution center,NaN,Karnataka,South,44086.0
41,WH036,Surat Distribution Center,NaN,Gujarat,West,43311.0


In [10]:
warehouses[['state', 'city']]
list(warehouses[['state', 'city']])

['state', 'city']

In [ ]:
city_map = (
    warehouses
    .dropna(subset=['state','city'])
    .drop_duplicates(subset='city')
    .set_index('state')['city']
)

In [24]:
state_map = (warehouses.drop_duplicates('state').set_index('state')['city'])

In [25]:
warehouses['city'] = warehouses['city'].fillna(warehouses['state'].map(state_map))

,warehouse_id,warehouse_name,city,state,region,capacity
19,WH028,Ahmedabad Distribution Center,NaN,Gujarat,West,49866.0
41,WH036,Surat Distribution Center,NaN,Gujarat,West,43311.0


In [29]:
warehouses['city'] = warehouses['city'].fillna('Ahmedabad')

In [31]:
warehouses['city'].isna().value_counts()

city
False    55
Name: count, dtype: int64

In [32]:
warehouses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   warehouse_id    55 non-null     object 
 1   warehouse_name  55 non-null     object 
 2   city            55 non-null     object 
 3   state           55 non-null     object 
 4   region          55 non-null     object 
 5   capacity        52 non-null     float64
dtypes: float64(1), object(5)
memory usage: 2.7+ KB


In [34]:
warehouses['warehouse_name']= warehouses['warehouse_name'].str.strip().str.title()

In [37]:
warehouses['region'] = warehouses['region'].str.strip().str.title()

In [39]:
warehouses['region'].value_counts().sum()

np.int64(55)

In [40]:
warehouses['capacity'] = warehouses['capacity'].fillna(warehouses.groupby(['region', 'state','city'])['capacity'].transform('median'))

In [ ]:
warehouses['capacity'] = warehouses['capacity'].fillna(warehouses['capacity'].median())

In [44]:
warehouses['capacity'].isna().value_counts()

capacity
False    55
Name: count, dtype: int64

In [46]:
warehouses['capacity'].max()

np.float64(49866.0)

In [47]:
# outlier remove
min_capacity = warehouses['capacity'].quantile(0.25)
max_capacity = warehouses['capacity'].quantile(0.75)

iqr_capacity = max_capacity - min_capacity

lower_capacity = min_capacity - 1.5 * iqr_capacity
higher_capacity = max_capacity + 1.5 * iqr_capacity

outlier_capacity = warehouses[
    (warehouses['capacity'] < lower_capacity) | 
    (warehouses['capacity'] > higher_capacity) 
]

warehouses = warehouses[
    (warehouses['capacity'] >= lower_capacity) & 
    (warehouses['capacity'] <= higher_capacity) 
]


In [48]:
warehouses = warehouses[warehouses['capacity'] >  0]

In [50]:
warehouses['capacity'].max()

np.float64(49866.0)

In [52]:
# sqlite 
conn = sqlite3.connect('managing.db')

In [54]:
warehouses.to_sql('warehouses',conn, if_exists='replace' , index=False)

50

In [55]:
pd.read_sql_query('select * from warehouses limit 1 ', conn)

,warehouse_id,warehouse_name,city,state,region,capacity
0,WH032,Faridabad Distribution Center,Faridabad,Haryana,North,18934.0


In [53]:
# SQL Questions

# Examples:

# Beginner
# Count warehouses by state.
# Average warehouse capacity.
# Largest warehouse.
# Intermediate
# Warehouses with missing capacity.
# Warehouses having invalid capacity.
# Duplicate warehouse names.
# Warehouses by region.
# Advanced
# Capacity contribution by region.
# Rank warehouses by capacity.
# Warehouses above regional average.
# Detect wrong city-state mappings.

In [ ]:
# Count warehouses by state.
pd.read_sql_query('select state, count(warehouse_id) as total_count from warehouses group by state order by total_count desc', conn)

In [ ]:
# Average warehouse capacity.
pd.read_sql_query('select warehouse_name, round(avg(capacity),2) as avg_capacity from warehouses group by warehouse_name order by avg_capacity desc ',conn)

In [60]:
# Largest warehouse.
pd.read_sql_query('select warehouse_name, capacity from warehouses where capacity = (select max(capacity) from warehouses)', conn)

,warehouse_name,capacity
0,Ahmedabad Distribution Center,49866.0


In [62]:
# Warehouses with missing capacity.
pd.read_sql_query('select * from warehouses where capacity is null', conn)

,warehouse_id,warehouse_name,city,state,region,capacity


In [63]:
# Warehouses having invalid capacity.
pd.read_sql_query('select * from warehouses where capacity is null or capacity < 0 ', conn)

,warehouse_id,warehouse_name,city,state,region,capacity


In [ ]:
# Duplicate warehouse names.
pd.read_sql_query('select warehouse_name , count(warehouse_name) as dupp_count from warehouses group by warehouse_name having count(warehouse_name) > 1 order by dupp_count desc', conn)

In [ ]:
# Warehouses by region.
pd.read_sql_query('select region , count(warehouse_name) as warehouses_count from warehouses group by region order by warehouses_count desc', conn)

In [ ]:
# Capacity contribution by region.
pd.read_sql_query("""
with contribution_data as (
    select sum(capacity) as total_capacity from warehouses
    )
    select region, sum(capacity) as capacity, 
    round(sum(capacity) * 100.0 / (select total_capacity from contribution_data ),2) as contribution_pct
    from warehouses
    group by region
    order by contribution_pct desc
""", conn)

In [ ]:
# Rank warehouses by capacity.
pd.read_sql_query('''
select warehouse_id,
    warehouse_name,
    capacity,
    dense_rank() over(order by capacity desc) as capacity_rank
    from warehouses''', conn)

In [ ]:
# Warehouses above regional average.
pd.read_sql_query(''' with regional_data as (
    select region, avg(capacity) as avg_regional_capacity
    from warehouses
    group by region
)
select 
w.warehouse_id,
w.warehouse_name,
w.region,
w.capacity
from warehouses w join regional_data r
on w.region = r.region
where w.capacity > avg_regional_capacity
''', conn)